# TalentGuard — Notebook 02: Pipeline de Limpieza y Preparación de Datos

**Proyecto:** TalentGuard: Sistema Inteligente para la Predicción del Riesgo de Rotación de Empleados  
**Autor:** Nicolas Gomez  
**Dataset:** IBM HR Analytics Employee Attrition & Performance  
**Objetivo:** Construir un pipeline reproducible que transforme el dataset crudo en un dataset limpio y listo para el modelado.

---

## Índice
1. Carga y diagnóstico inicial
2. Eliminación de columnas constantes
3. Tratamiento de valores nulos
4. Eliminación de duplicados
5. Codificación de variables categóricas
6. Verificación del dataset procesado
7. Separación train / test
8. Guardado del dataset procesado
9. Resumen del pipeline

---
## Importaciones

In [331]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Semilla fija para reproducibilidad
RANDOM_STATE = 42

# Rutas
RAW_PATH = Path('../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


---
## 1. Carga y Diagnóstico Inicial

In [332]:
df = pd.read_csv(RAW_PATH)

print(f'Shape: {df.shape}')
print(f'Filas: {df.shape[0]} | Columnas: {df.shape[1]}')

Shape: (1470, 35)
Filas: 1470 | Columnas: 35


In [333]:
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [334]:
df.dtypes

Age                         int64
Attrition                     str
BusinessTravel                str
DailyRate                   int64
Department                    str
DistanceFromHome            int64
Education                   int64
EducationField                str
EmployeeCount               int64
EmployeeNumber              int64
EnvironmentSatisfaction     int64
Gender                        str
HourlyRate                  int64
JobInvolvement              int64
JobLevel                    int64
JobRole                       str
JobSatisfaction             int64
MaritalStatus                 str
MonthlyIncome               int64
MonthlyRate                 int64
NumCompaniesWorked          int64
Over18                        str
OverTime                      str
PercentSalaryHike           int64
PerformanceRating           int64
RelationshipSatisfaction    int64
StandardHours               int64
StockOptionLevel            int64
TotalWorkingYears           int64
TrainingTimesL

In [335]:
# Diagnóstico de nulos
nulos = df.isnull().sum()
print('Valores nulos por columna:')
print(nulos[nulos > 0] if nulos.sum() > 0 else 'No hay valores nulos ✅')

Valores nulos por columna:
No hay valores nulos ✅


In [336]:
# Diagnóstico de duplicados
duplicados = df.duplicated().sum()
print(f'Registros duplicados: {duplicados}')
if duplicados == 0:
    print('No hay duplicados ✅')

Registros duplicados: 0
No hay duplicados ✅


In [337]:
# Estadísticas descriptivas básicas
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Age,1470.0,NaN,NaN,NaN,36.92381,9.135373,18.0,30.0,36.0,43.0,60.0
Attrition,1470,2,No,1233,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BusinessTravel,1470,3,Travel_Rarely,1043,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DailyRate,1470.0,NaN,NaN,NaN,802.485714,403.5091,102.0,465.0,802.0,1157.0,1499.0
Department,1470,3,Research & Development,961,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DistanceFromHome,1470.0,NaN,NaN,NaN,9.192517,8.106864,1.0,2.0,7.0,14.0,29.0
Education,1470.0,NaN,NaN,NaN,2.912925,1.024165,1.0,2.0,3.0,4.0,5.0
EducationField,1470,6,Life Sciences,606,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EmployeeCount,1470.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
EmployeeNumber,1470.0,NaN,NaN,NaN,1024.865306,602.024335,1.0,491.25,1020.5,1555.75,2068.0


**Diagnóstico inicial:**
- El dataset tiene 1.470 registros y 35 columnas, confirmado.
- No hay valores nulos ni duplicados, lo que simplifica el pipeline.
- Se identifican columnas constantes que deben eliminarse antes del modelado.

---
## 2. Eliminación de Columnas Constantes

**Justificación:** Las columnas `EmployeeCount`, `Over18` y `StandardHours` tienen un único valor en todos los registros, por lo que no aportan ninguna capacidad predictiva al modelo y generan ruido innecesario.

In [338]:
# Verificar que son realmente constantes
columnas_constantes = ['EmployeeCount', 'Over18', 'StandardHours']
for col in columnas_constantes:
    print(f'{col}: valores únicos → {df[col].unique()}')

EmployeeCount: valores únicos → [1]
Over18: valores únicos → <StringArray>
['Y']
Length: 1, dtype: str
StandardHours: valores únicos → [80]


In [339]:
# También eliminamos EmployeeNumber: es un identificador sin valor predictivo
columnas_a_eliminar = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']

df = df.drop(columns=columnas_a_eliminar)

print(f'Columnas eliminadas: {columnas_a_eliminar}')
print(f'Shape tras eliminación: {df.shape}')

Columnas eliminadas: ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
Shape tras eliminación: (1470, 31)


---
## 3. Tratamiento de Valores Nulos

**Justificación:** El diagnóstico confirmó que el dataset no tiene valores nulos. Esta celda verifica el estado una vez más tras las transformaciones previas y documenta el resultado.

In [340]:
nulos_post = df.isnull().sum().sum()
print(f'Total de valores nulos tras limpieza: {nulos_post}')
print('No se requiere imputación.')

Total de valores nulos tras limpieza: 0
No se requiere imputación.


---
## 4. Eliminación de Duplicados

In [341]:
df = df.drop_duplicates()
print(f'Shape tras eliminación de duplicados: {df.shape}')
print('No se eliminaron registros.')

Shape tras eliminación de duplicados: (1470, 31)
No se eliminaron registros.


---
## 5. Codificación de Variables Categóricas

El dataset contiene las siguientes variables categóricas que requieren transformación:

| Variable | Tipo | Estrategia |
|---|---|---|
| `Attrition` | Binaria (Yes/No) | Label Encoding → 1/0 |
| `OverTime` | Binaria (Yes/No) | Label Encoding → 1/0 |
| `Gender` | Binaria (Male/Female) | Label Encoding → 1/0 |
| `Department` | Nominal (3 categorías) | One-Hot Encoding |
| `BusinessTravel` | Ordinal (3 categorías) | Ordinal Encoding manual |
| `EducationField` | Nominal (6 categorías) | One-Hot Encoding |
| `JobRole` | Nominal (9 categorías) | One-Hot Encoding |
| `MaritalStatus` | Nominal (3 categorías) | One-Hot Encoding |

**Justificación de la estrategia:**
- Variables binarias: Label Encoding es suficiente y evita dimensionalidad innecesaria.
- Variables nominales sin orden natural: One-Hot Encoding para no introducir jerarquías artificiales.
- `BusinessTravel`: tiene un orden implícito (No Travel < Travel Rarely < Travel Frequently), se codifica como ordinal.

In [342]:
# Verificar variables categóricas actuales
categoricas = df.select_dtypes(include='str').columns.tolist()
print(f'Variables categóricas: {categoricas}')
for col in categoricas:
    print(f'  {col}: {df[col].unique()}')

Variables categóricas: ['Attrition', 'BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']
  Attrition: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
  BusinessTravel: <StringArray>
['Travel_Rarely', 'Travel_Frequently', 'Non-Travel']
Length: 3, dtype: str
  Department: <StringArray>
['Sales', 'Research & Development', 'Human Resources']
Length: 3, dtype: str
  EducationField: <StringArray>
[   'Life Sciences',            'Other',          'Medical',
        'Marketing', 'Technical Degree',  'Human Resources']
Length: 6, dtype: str
  Gender: <StringArray>
['Female', 'Male']
Length: 2, dtype: str
  JobRole: <StringArray>
[          'Sales Executive',        'Research Scientist',
     'Laboratory Technician',    'Manufacturing Director',
 'Healthcare Representative',                   'Manager',
      'Sales Representative',         'Research Director',
           'Human Resources']
Length: 9, dtype: str
  MaritalStatus: <StringArray>
['S

In [343]:
# --- 5.1 Codificación binaria ---

# Attrition: Yes=1, No=0 (variable objetivo)
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# OverTime: Yes=1, No=0
df['OverTime'] = df['OverTime'].map({'Yes': 1, 'No': 0})

# Gender: Male=1, Female=0
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

print('Codificación binaria aplicada')
print(f"  Attrition: {df['Attrition'].value_counts().to_dict()}")
print(f"  OverTime:  {df['OverTime'].value_counts().to_dict()}")
print(f"  Gender:    {df['Gender'].value_counts().to_dict()}")

Codificación binaria aplicada
  Attrition: {0: 1233, 1: 237}
  OverTime:  {0: 1054, 1: 416}
  Gender:    {1: 882, 0: 588}


In [344]:
# --- 5.2 Codificación ordinal para BusinessTravel ---
# Orden: Non-Travel=0, Travel_Rarely=1, Travel_Frequently=2

travel_order = {'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}
df['BusinessTravel'] = df['BusinessTravel'].map(travel_order)

print('Codificación ordinal BusinessTravel')
print(df['BusinessTravel'].value_counts().to_dict())

Codificación ordinal BusinessTravel
{1: 1043, 2: 277, 0: 150}


In [345]:
# --- 5.3 One-Hot Encoding para variables nominales ---
# drop_first=True para evitar multicolinealidad perfecta

columnas_ohe = ['Department', 'EducationField', 'JobRole', 'MaritalStatus']

df = pd.get_dummies(df, columns=columnas_ohe, drop_first=True)

print(f'One-Hot Encoding aplicado')
print(f'Shape tras OHE: {df.shape}')
print(f'Columnas creadas: {[c for c in df.columns if any(c.startswith(p) for p in columnas_ohe)]}')

One-Hot Encoding aplicado
Shape tras OHE: (1470, 44)
Columnas creadas: ['Department_Research & Development', 'Department_Sales', 'EducationField_Life Sciences', 'EducationField_Marketing', 'EducationField_Medical', 'EducationField_Other', 'EducationField_Technical Degree', 'JobRole_Human Resources', 'JobRole_Laboratory Technician', 'JobRole_Manager', 'JobRole_Manufacturing Director', 'JobRole_Research Director', 'JobRole_Research Scientist', 'JobRole_Sales Executive', 'JobRole_Sales Representative', 'MaritalStatus_Married', 'MaritalStatus_Single']


In [346]:
# Verificar que no quedan variables categóricas sin codificar
restantes = df.select_dtypes(include='str').columns.tolist()
print(f'Variables categóricas restantes: {restantes if restantes else "Ninguna"}')

Variables categóricas restantes: Ninguna


In [347]:
# Convertir columnas booleanas generadas por OHE a int (compatibilidad con sklearn)
bool_cols = df.select_dtypes(include='bool').columns.tolist()
df[bool_cols] = df[bool_cols].astype(int)
print(f'Columnas booleanas convertidas a int: {len(bool_cols)}')

Columnas booleanas convertidas a int: 17


---
## 6. Verificación del Dataset Procesado

In [348]:
print(f'Shape final del dataset procesado: {df.shape}')
print(f'Tipos de datos: {df.dtypes.value_counts().to_dict()}')
print(f'Valores nulos: {df.isnull().sum().sum()}')
df.head(3)

Shape final del dataset procesado: (1470, 44)
Tipos de datos: {dtype('int64'): 44}
Valores nulos: 0


,Age,Attrition,BusinessTravel,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,...,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single
0,41,1,1,1102,1,2,2,0,94,3,...,0,0,0,0,0,0,1,0,0,1
1,49,0,2,279,8,1,3,1,61,2,...,0,0,0,0,0,1,0,0,1,0
2,37,1,1,1373,2,2,4,1,92,2,...,0,1,0,0,0,0,0,0,0,1


In [349]:
# Distribución de la variable objetivo tras codificación
conteo = df['Attrition'].value_counts()
pct = df['Attrition'].value_counts(normalize=True) * 100

print('Distribución de Attrition en el dataset procesado:')
print(f'  No (0): {conteo[0]} registros ({pct[0]:.1f}%)')
print(f'  Yes (1): {conteo[1]} registros ({pct[1]:.1f}%)')
print(f'  → Desbalance confirmado: {pct[0]:.1f}% / {pct[1]:.1f}%')

Distribución de Attrition en el dataset procesado:
  No (0): 1233 registros (83.9%)
  Yes (1): 237 registros (16.1%)
  → Desbalance confirmado: 83.9% / 16.1%


---
## 7. Separación Train / Test

**Justificación:**
- Se usa `stratify=y` para preservar la proporción de clases (16% / 84%) en ambos conjuntos.
- `random_state=42` garantiza reproducibilidad.
- Split 80/20: suficiente para un dataset de 1.470 registros.

> **Nota sobre el orden de las transformaciones:**  
> La codificación de variables categóricas (label encoding, ordinal y OHE) se aplicó sobre el dataset completo **antes** del split. Esto es válido en este caso porque todas las categorías posibles están presentes en el dataset y no existe riesgo de categorías "no vistas" en test. Sin embargo, el **escalado de variables numéricas** (StandardScaler / MinMaxScaler) se realizará en el notebook 03, ajustándose **exclusivamente sobre `X_train`** y aplicándose sobre `X_test`. Esto evita la fuga de datos (*data leakage*), donde el modelo aprendería indirectamente la distribución del conjunto de prueba.

In [350]:
# Separar features y variable objetivo
X = df.drop(columns=['Attrition'])
y = df['Attrition']

print(f'Features (X): {X.shape}')
print(f'Variable objetivo (y): {y.shape}')

Features (X): (1470, 43)
Variable objetivo (y): (1470,)


In [351]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y  # preserva proporción de clases
)

print(f'Train: {X_train.shape[0]} registros ({X_train.shape[0]/len(df)*100:.1f}%)')
print(f'Test:  {X_test.shape[0]} registros ({X_test.shape[0]/len(df)*100:.1f}%)')
print()
print('Distribución de clases en Train:')
print(y_train.value_counts(normalize=True).round(3))
print()
print('Distribución de clases en Test:')
print(y_test.value_counts(normalize=True).round(3))

Train: 1176 registros (80.0%)
Test:  294 registros (20.0%)

Distribución de clases en Train:
Attrition
0    0.838
1    0.162
Name: proportion, dtype: float64

Distribución de clases en Test:
Attrition
0    0.84
1    0.16
Name: proportion, dtype: float64


---
## 8. Guardado del Dataset Procesado

In [352]:
# Dataset completo procesado (para el dashboard)
df.to_csv(PROCESSED_DIR / 'dataset_limpio.csv', index=False)

# Conjuntos de entrenamiento y prueba
X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DIR / 'X_test.csv', index=False)
y_train.to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_csv(PROCESSED_DIR / 'y_test.csv', index=False)

print('Archivos guardados en data/processed/ ✅')
print(f'  dataset_limpio.csv  → {df.shape}')
print(f'  X_train.csv         → {X_train.shape}')
print(f'  X_test.csv          → {X_test.shape}')
print(f'  y_train.csv         → {y_train.shape}')
print(f'  y_test.csv          → {y_test.shape}')

Archivos guardados en data/processed/ ✅
  dataset_limpio.csv  → (1470, 44)
  X_train.csv         → (1176, 43)
  X_test.csv          → (294, 43)
  y_train.csv         → (1176,)
  y_test.csv          → (294,)


---
## 9. Resumen del Pipeline

| Paso | Acción | Resultado |
|---|---|---|
| 1. Carga | Lectura del CSV crudo | 1.470 filas × 35 columnas |
| 2. Columnas constantes | Eliminación de `EmployeeCount`, `Over18`, `StandardHours`, `EmployeeNumber` | −4 columnas |
| 3. Nulos | Sin imputación requerida | 0 nulos |
| 4. Duplicados | Sin eliminación requerida | 0 duplicados |
| 5a. Codificación binaria | `Attrition`, `OverTime`, `Gender` → 0/1 | Sin nuevas columnas |
| 5b. Codificación ordinal | `BusinessTravel` → 0/1/2 | Sin nuevas columnas |
| 5c. One-Hot Encoding | `Department`, `EducationField`, `JobRole`, `MaritalStatus` | +columnas dummies |
| 6. Separación | 80% train / 20% test con stratify | 1.176 train / 294 test |
| 7. Guardado | 5 archivos CSV en `data/processed/` | ✅ |

**Dataset final:** ~1.470 filas × ~46 columnas (varía según categorías OHE)

**Nota sobre escalado:** El escalado de variables numéricas (StandardScaler / MinMaxScaler) se realizará dentro del notebook de modelado (`03_modelado.ipynb`) usando un Pipeline de sklearn, ajustándose **solo sobre X_train** y aplicándose sobre X_test. Esto garantiza la ausencia de fuga de datos.